In [ ]:
%%bash
### subset UHVDB and run mcl clustering on subsets
# seqkit fx2tab \
#     ../uhvdb_clustering/uhvdb.rmdup.fna.gz \
#     --only-id --name --no-qual \
#     --out-file uhgv.ids.tsv

# for rep in {5..5}; do
#     for subset in "400000"; do
#         mkdir -p subset_${subset}/

#         shuf --random-source=<(yes $rep) -n $subset \
#             uhgv.ids.tsv \
#             > subset_${subset}/uhgv_hq_plus.subset${subset}_rep${rep}_ids.tsv

#         csvtk join --tabs \
#             -f "1;1" \
#             ../uhvdb_clustering/vclust/uhvdb.vclust_mcl.tsv \
#             subset_${subset}/uhgv_hq_plus.subset${subset}_rep${rep}_ids.tsv | \
#         csvtk join --tabs \
#             -f "2;1" \
#             - subset_${subset}/uhgv_hq_plus.subset${subset}_rep${rep}_ids.tsv \
#             --out-file subset_${subset}/uhgv_hq_plus.subset${subset}_rep${rep}_gani.tsv

#         micromamba run -n mcl \
#             mcl \
#                 subset_${subset}/uhgv_hq_plus.subset${subset}_rep${rep}_gani.tsv \
#                 --abc \
#                 -sort revsize \
#                 -te 32 \
#                 -o subset_${subset}/uhgv_hq_plus.subset${subset}_rep${rep}.mcl
#     done
# done

In [ ]:
### function to load mcl clusters
import polars as pl

uhvdb_final_ids = set(
    pl.read_csv('../uhvdb_clustering/uhvdb_final_ids.tsv', has_header=False, new_columns=['seq_name'])['seq_name']
)

def load_mcl_clusters(mcl, seq_id):
    # assign sequences to mcl clusters
    clusters = {}

    cluster_id = 0
    with open(mcl, 'r') as mcl_file:
        for line in mcl_file:
            cluster_id += 1
            for node in line.strip().split():
                if node in uhvdb_final_ids:
                    clusters[node] = cluster_id

    # assign unclustered sequences to their own cluster
    with open(seq_id, 'r') as seqid_file:
        for line in seqid_file:
            sequence = line.strip().split()[0]
            if sequence not in clusters and sequence in uhvdb_final_ids:
                cluster_id += 1
                clusters[sequence] = cluster_id

    # convert to a DataFrame
    clusters_df = pl.DataFrame({
        'seq_id': list(clusters.keys()),
        'cluster_id': list(clusters.values())
    })

    return clusters_df